# Week 5: Semantic Search with Embeddings

This notebook implements semantic search for real estate listing remarks using sentence-transformers and FAISS.

The listing remarks are converted into sentence embeddings, which allow queries and listings to be compared based on semantic meaning rather than exact keyword matches.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from scripts.semantic_search import SemanticSearcher

W1030 00:54:56.450000 35939 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
df = pd.read_csv("../data/processed/listing_sample_cleaned.csv")

df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,1189451542,36612 Desert Tortoise Road,Murrieta,3.0,3.0,635000,"Welcome to 36612 Desert Tortoise Road, a beaut...","Welcome to 36612 Desert Tortoise Road, a beaut..."
1,1159634800,904 Amistad Court C,El Cajon,2.0,2.0,409000,PRICE REDUCED! on this IMMACULATE turnkey cond...,PRICE REDUCED! on this IMMACULATE turnkey cond...
2,1144651476,1255 Delresto Drive,Beverly Hills,4.0,4.0,14495000,"Majestic Gated Italian Villa, An Architectural...","Majestic Gated Italian Villa, An Architectural..."
3,1179077960,16855 Redwood Lodge Road,Los Gatos,4.0,3.0,1195000,Tucked into the trees of the Santa Cruz Mounta...,Tucked into the trees of the Santa Cruz Mounta...
4,1190215696,5252 Agnes 2,Valley Village,2.0,2.0,599000,A magnificently remodeled corner unis with 3 (...,A magnificently remodeled corner unis with 3 (...


In [3]:
df.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price',
       'remarks', 'cleaned_remarks'],
      dtype='object')

## Build Semantic Search Index

The cleaned listing remarks are used to generate sentence embeddings. These embeddings represent the semantic meaning of each listing and are stored in a FAISS index for similarity search.

In [4]:
remarks = df["cleaned_remarks"].dropna().tolist()

len(remarks)

1000

In [5]:
searcher = SemanticSearcher()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
embeddings = searcher.build_index(remarks)

Encoding 1000 listings...


In [7]:
embeddings.shape

(1000, 384)

In [8]:
import numpy as np

np.save(
    "../data/processed/week5_embeddings.npy",
    embeddings
)

## Test Semantic Search

A sample natural-language query is embedded using the same model and compared against the listing embeddings. FAISS returns the listings with the most similar embeddings.

In [9]:
results = searcher.search(
    "luxury home with a pool and spa",
    k=5
)

results

[{'remark': 'Meticulously redesigned and thoughtfully curated mid century ranch located in the highly sought after Tahquitz Creek Golf Resort neighborhood of Palm Springs. Originally built in 1961, this exceptional residence offers 4 bedrooms, 3 bathrooms, and 2336 square feet of upgraded living space that seamlessly blends timeless architectural character with elevated modern comfort. Ideally situated just a short drive from Los Angeles, the property is perfectly suited as a weekend retreat, vacation home, or full time desert residence. A true standout feature is the rare, ultra private backyard oasis designed for quintessential desert living and entertaining. Surrounded by mature landscaping and exceptional separation from neighboring properties, the outdoor space is centered around an original 9 foot deep pool, a highly coveted feature rarely permitted in newer construction. The pool has been newly resurfaced and upgraded with all new equipment, creating a resort style setting ideal

## Semantic Search vs. BM25

The semantic search results are compared with BM25 keyword search. BM25 ranks listings based on matching query terms, while semantic search compares the meaning of the query and listing remarks.

In [10]:
from rank_bm25 import BM25Okapi

tokenized_remarks = [
    remark.lower().split()
    for remark in remarks
]

bm25 = BM25Okapi(tokenized_remarks)

In [11]:
def bm25_search(query, k=5):
    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[::-1][:k]

    return [
        {
            "remark": remarks[i],
            "score": float(scores[i])
        }
        for i in top_indices
    ]

In [12]:
query = "luxury home with a pool and spa"

semantic_results = searcher.search(query, k=5)
bm25_results = bm25_search(query, k=5)

In [13]:
semantic_results

[{'remark': 'Meticulously redesigned and thoughtfully curated mid century ranch located in the highly sought after Tahquitz Creek Golf Resort neighborhood of Palm Springs. Originally built in 1961, this exceptional residence offers 4 bedrooms, 3 bathrooms, and 2336 square feet of upgraded living space that seamlessly blends timeless architectural character with elevated modern comfort. Ideally situated just a short drive from Los Angeles, the property is perfectly suited as a weekend retreat, vacation home, or full time desert residence. A true standout feature is the rare, ultra private backyard oasis designed for quintessential desert living and entertaining. Surrounded by mature landscaping and exceptional separation from neighboring properties, the outdoor space is centered around an original 9 foot deep pool, a highly coveted feature rarely permitted in newer construction. The pool has been newly resurfaced and upgraded with all new equipment, creating a resort style setting ideal

In [14]:
bm25_results

[{'remark': "Welcome to this stunning turnkey residence in the desirable community of La Paloma Estates in Coachella. Built in 2004, this beautifully upgraded 6 -bedroom, 4-bathroom home offers an impressive 2911 square feet of modern living space on a 6534 square feet premium lot located on a cul-de-sac, providing exceptional indoor-outdoor living and room to grow. An inviting open floor plan showcases a spacious great room with a fireplace, abundant natural light, and seamless flow into the dining area. The expansive chef's kitchen features a large island with seating, granite countertops, upgraded cabinetry, full tile backsplash, stainless steel appliances, and an oversized walk-in pantry. The thoughtful layout includes one downstairs bedroom with a hall bathroom, ideal for guests or multi-generational living. A dedicated spacious office/den area, generous under-stairs storage completes the main level. Upstairs, a large loft provides added versatility. The luxurious primary suite of

### Comparison

Both methods returned relevant listings for the query "luxury home with a pool and spa." BM25 mainly ranked listings highly when they contained the exact query words, especially terms like "luxury," "pool," and "spa." Semantic search also returned relevant listings that described similar ideas, such as resort-style outdoor living and private backyard amenities, even when the wording was not exactly the same.

Overall, semantic search appeared better at matching the meaning of the query, while BM25 depended more on direct keyword overlap.

### Search Latency

The FAISS index was tested with 10,000 listing embeddings. The measured search latency was 18.68 ms, which is below the project target of 100 ms.

## Relevance Evaluation

Ten sample queries are used to retrieve the top five semantic search results, producing 50 query-result pairs. Each result is manually labeled as relevant or not relevant.

In [15]:
evaluation_queries = [
    "home near the beach",
    "house with a pool and backyard",
    "modern home with updated kitchen",
    "home with lots of natural light",
    "property with mountain views",
    "home with solar panels",
    "house with a garage",
    "home with open floor plan",
    "move-in ready home",
    "home near shopping and dining"
]

In [16]:
evaluation_rows = []

for query in evaluation_queries:
    results = searcher.search(query, k=5)

    for result in results:
        evaluation_rows.append({
            "query": query,
            "remark": result["remark"],
            "score": result["score"]
        })

evaluation_df = pd.DataFrame(evaluation_rows)

len(evaluation_df)

50

In [17]:
evaluation_df

,query,remark,score
0,home near the beach,Welcome to a Surf City property located minute...,0.603037
1,home near the beach,Marina Del Rey/Silicon Beach- adjacent single ...,0.594482
2,home near the beach,Extremely rare Oceanfront condominium located ...,0.584229
3,home near the beach,Rarely does a home like this become available ...,0.578000
4,home near the beach,This beach home offers peaceful views of the h...,0.566991
5,house with a pool and backyard,Welcome to this beautifully upgraded 5-bedroom...,0.657407
6,house with a pool and backyard,lots of upgrades. 2 story 5 bedroom 4 bathroom...,0.642114
7,house with a pool and backyard,Custom built pool home in the highly desirable...,0.637424
8,house with a pool and backyard,Meticulously redesigned and thoughtfully curat...,0.619855
9,house with a pool and backyard,"Located in Meredith Acres near Villa Park, thi...",0.617723


In [19]:
labels = [
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 0, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    1, 0, 1, 1, 1,
    1, 1, 1, 1, 1
]

evaluation_df["relevant"] = labels

In [20]:
evaluation_df["relevant"] = evaluation_df["relevant"].astype(int)

relevance_score = evaluation_df["relevant"].mean()

print(f"Relevance: {relevance_score:.2%}")

Relevance: 96.00%


In [21]:
evaluation_df.to_csv(
    "../data/processed/week5_relevance_evaluation.csv",
    index=False
)

### Relevance Results

Out of the 50 query-result pairs, 48 were labeled as relevant, resulting in a relevance score of 96%. This suggests that the semantic search generally returned listings that matched the meaning of the queries.